<a href="https://colab.research.google.com/github/jeehoh/deep-learning-playground/blob/main/ch3_applying_tensorflow_knowledge_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf

In [ ]:
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

In [ ]:
train_images = tf.constant(train_images)
train_labels = tf.constant(train_labels)
test_images = tf.constant(test_images)
test_labels = tf.constant(test_labels)

In [ ]:
print(f"{train_images.shape=}")
print(f"{train_labels.shape=}")
print(f"{train_images.dtype=}")
print(f"{train_labels.dtype=}")
print(f"{test_images.shape=}")
print(f"{test_labels.shape=}")
print(f"{test_images.dtype=}")
print(f"{test_labels.dtype=}")

In [ ]:
train_images = tf.reshape(train_images, shape=(60_000, 28 * 28))
train_images = tf.cast(train_images, dtype="float32") / 255.0
test_images = tf.reshape(test_images, shape=(10_000, 28 * 28))
test_images = tf.cast(test_images, dtype="float32") / 255.0
train_labels = tf.cast(train_labels, dtype="int64")
test_labels = tf.cast(test_labels, dtype="int64")

In [ ]:
print(f"{train_images.shape=}")
print(f"{train_labels.shape=}")
print(f"{train_images.dtype=}")
print(f"{train_labels.dtype=}")
print(f"{test_images.shape=}")
print(f"{test_labels.shape=}")
print(f"{test_images.dtype=}")
print(f"{test_labels.dtype=}")

In [ ]:
class Dense:
  def __init__(self, input_size, output_size, activation=None):
    self.W = tf.Variable(tf.random.normal(shape=(input_size, output_size)))
    self.b = tf.Variable(tf.zeros(shape=(output_size,)))
    self.activation = activation

  def __call__(self, inputs):
    x = tf.matmul(inputs, self.W)
    x = x + self.b
    if self.activation is not None:
      x = self.activation(x)
    return x

  @property
  def weights(self):
    return [self.W, self.b]

In [ ]:
test_dense = Dense(28 * 28, 512)
assert len(test_dense.weights) == 2
assert test_dense.weights[0].shape == (28 * 28, 512)
assert test_dense.weights[1].shape == (512,)

In [ ]:
class Layer:
  def __init__(self, layers):
    self.layers = layers

  def __call__(self, inputs):
    x = inputs
    for layer in self.layers:
      x = layer(x)
    return x

  @property
  def weights(self):
    weights = []
    for layer in self.layers:
      weights += layer.weights
    return weights

In [ ]:
test_layer = Layer([
    Dense(28 * 28, 512, activation=tf.nn.relu),
    Dense(512, 10, activation=tf.nn.softmax)
])
assert len(test_layer.weights) == 4
assert test_layer.weights[0].shape == (28 * 28, 512)
assert test_layer.weights[1].shape == (512,)
assert test_layer.weights[2].shape == (512, 10)
assert test_layer.weights[3].shape == (10,)

In [ ]:
import math

class BatchGenerator:
  def __init__(self, inputs, labels, batch_size=128):
    assert inputs.shape[0] == labels.shape[0]
    self.index = 0
    self.inputs = inputs
    self.labels = labels
    self.batch_size = batch_size
    self.num_batches = math.ceil(inputs.shape[0] / batch_size)

  def next(self):
    inputs_batch = self.inputs[self.index : self.index + self.batch_size]
    labels_batch = self.labels[self.index : self.index + self.batch_size]
    self.index = self.index + self.batch_size
    return inputs_batch, labels_batch

In [ ]:
test_batch_gen = BatchGenerator(train_images, train_labels)
assert test_batch_gen.num_batches == 469
train_images_batch, train_labels_batch = test_batch_gen.next()
assert len(train_images_batch) == 128
assert len(train_labels_batch) == 128

In [ ]:
learning_rate = 1e-3

@tf.function(jit_compile=True)
def update_weights(gradients, weights):
  for g, w in zip(gradients, weights):
    w.assign(w - learning_rate * g)

In [ ]:
@tf.function(jit_compile=True)
def one_training_step(model, input_batch, label_batch):
  with tf.GradientTape() as tape:
    predictions = model(input_batch)
    loss = tf.nn.sparse_softmax_cross_entropy_with_logits(label_batch, predictions)
    avg_loss = tf.reduce_mean(loss)
  grad_loss_wrt_weights = tape.gradient(avg_loss, model.weights)
  update_weights(grad_loss_wrt_weights, model.weights)
  return avg_loss

In [ ]:
def fit(model, images, labels, epochs, batch_size=128):
    for epoch_counter in range(epochs):
        print(f"Epoch {epoch_counter}")
        batch_generator = BatchGenerator(images, labels, batch_size)
        for batch_counter in range(batch_generator.num_batches):
            images_batch, labels_batch = batch_generator.next()
            loss = one_training_step(model, images_batch, labels_batch)
            if batch_counter % 100 == 0:
                print(f"loss at batch {batch_counter}: {loss:.2f}")

In [ ]:
model = Layer([
    Dense(28 * 28, 512, activation=tf.nn.relu),
    Dense(512, 10)
])

In [ ]:
fit(model, train_images, train_labels, epochs=10, batch_size=128)

In [ ]:
predictions = model(test_images)
predicted_labels = tf.math.argmax(predictions, axis=1)
matches = predicted_labels == test_labels
matches = tf.cast(matches, dtype='float32')
print(f"accuracy: {tf.reduce_mean(matches):.2f}")